# LINCS L1000 drugs on the Figure 5 meta-gene modules (MGMs)

💡 **Environment:** `clamp-analyses`

View-only report of rule `crispercas_metalv_l1000` (`scripts/archs4/crispercas/l1000_metalv.py`).
Each LINCS L1000 consensus drug signature (1,170 DrugBank drugs) is scored on each MGM's top 1% of gene loadings
(185 genes; those measured in LINCS): the mean drug z-score over these genes, standardized per drug across the
27 GSEA-significant MGMs (the PhenoPLIER set). Drugs are ranked by descending score (module induced).
Significance: 10,000 random LINCS gene sets of the same size, one-sided permutation P, BH FDR across drugs.
The gene-permutation null ignores co-expression among module genes, so FDR is lenient; rank and permutation z
are the more informative quantities.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    RANKING = Path(snakemake.input.ranking)
except NameError:
    RANKING = Path('../../../../output/03_model_biology/02_archs4/04_crispercas/03_metalv/l1000/l1000_fig5_ranking.csv')
r = pd.read_csv(RANKING)
r['MGM'] = r.metaLV.str.replace('metaLV', 'MGM')
ORDER = ['MGM91', 'MGM588', 'MGM175', 'MGM335']
PICK = {'MGM91': 'Clomifene', 'MGM588': 'Clomifene', 'MGM175': 'Cortisone acetate', 'MGM335': 'Calcitriol'}
assert set(r.MGM) == set(ORDER) and r.groupby('MGM').size().eq(1170).all()

## Mechanism-linked drug classes

Drugs expected to induce each program, defined from its biology (flagged in the tables below):
- MGM91, MGM588 (cholesterol biosynthesis / SREBP): statins, CYP51 azoles, post-squalene sterol-enzyme inhibitors and SREBP2-activating antipsychotics/antidepressants (ATC N05A, N06A).
- MGM175 (hepatocyte, complement and coagulation): glucocorticoids/corticosteroids, bile acids, fibrates, glitazones, estrogens.
- MGM335 (macrophage activation): vitamin D and retinoids (myeloid differentiation), glucocorticoids, NSAIDs, immunosuppressants.

In [ ]:
def has(s, *keys):
    return s.fillna('').str.contains('|'.join(keys), case=False, regex=True)

sterol = ['statin$', 'conazole$', 'Clomifene', 'Tamoxifen', 'Amiodarone', 'Haloperidol', 'Trazodone', 'Aripiprazole', 'Trimipramine']
LINK = {
    'sterol': lambda d: has(d.drug, *sterol) & ~has(d.drug, 'Bestatin', 'Cilastatin', 'Wiskostatin', 'blebbistatin', 'Alrestatin')
                        | has(d.atc_codes, 'N05A', 'N06A'),
    'MGM175': lambda d: has(d.categories, 'Glucocorticoid', 'Corticosteroid') | has(d.drug, 'cholic', 'Ursodeoxycholic', 'fibrate$', 'Gemfibrozil', 'glitazone', 'PREGN-4-ENE')
                        | has(d.atc_codes, 'G03C'),
    'MGM335': lambda d: has(d.drug, 'Calcitriol', 'DN-101', 'Vitamin A', 'Tretinoin') | has(d.categories, 'Glucocorticoid', 'Anti-Inflammatory Agents, Non-Steroidal')
                        | has(d.atc_codes, 'L04'),
}
r['linked'] = False
for m in ORDER:
    i = r.MGM == m
    r.loc[i, 'linked'] = LINK['sterol' if m in ('MGM91', 'MGM588') else m](r[i]).values

## Selected drug per MGM (Figure 5)

In [ ]:
picks = pd.concat([r[(r.MGM == m) & (r.drug == d)] for m, d in PICK.items()])
summary = r.groupby('MGM').agg(n_genes=('n_genes', 'first'), n_fdr_0_05=('fdr', lambda q: (q < 0.05).sum()),
                               linked_top10=('linked', lambda s: s.iloc[:10].sum()),
                               linked_top1pct=('linked', lambda s: s.iloc[:12].sum()))
display(picks.set_index('MGM')[['drug', 'rank', 'score', 'perm_z', 'perm_p', 'fdr']].join(summary).loc[ORDER])

## Top 25 drugs per MGM

In [ ]:
cols = ['rank', 'drug', 'drugbank_id', 'score', 'mean_z', 'perm_z', 'fdr', 'linked']
for m in ORDER:
    print(m)
    display(r[r.MGM == m].head(25)[cols].style.format({'score': '{:.2f}', 'mean_z': '{:.2f}', 'perm_z': '{:.1f}', 'fdr': '{:.1e}'})
            .apply(lambda row: ['font-weight: bold' if row.drug == PICK[m] else '' for _ in row], axis=1).hide(axis='index'))

## Score by rank (mechanism-linked drugs in colour, Figure 5 drug labelled)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2), sharey=False)
for ax, m in zip(axes, ORDER):
    d = r[r.MGM == m]
    ax.scatter(d['rank'], d.score, s=3, c='#BBBBBB')
    ax.scatter(d[d.linked]['rank'], d[d.linked].score, s=8, c='#2A7AB0', label='mechanism-linked')
    p = d[d.drug == PICK[m]].iloc[0]
    ax.scatter(p['rank'], p.score, s=30, facecolors='none', edgecolors='#C0392B')
    ax.annotate(f"{p.drug} (rank {p['rank']})", (p['rank'], p.score), xytext=(60, -4), textcoords='offset points', fontsize=8,
                arrowprops=dict(arrowstyle='-', lw=0.5))
    ax.set(title=m, xlabel='rank (descending score)', xscale='log')
axes[0].set_ylabel('score (z across 27 MGMs)')
axes[0].legend(fontsize=7, frameon=False)
plt.tight_layout()
plt.show()